# A100에서 같은 조건의 TFT 3개 seed 쌍을 새로 학습

**설치 상태: 작성 시 실제 Colab 설치·학습은 아직 검증하지 않았습니다.** 아래 설치/사전 검사를 통과한 실행만 실험으로 기록합니다.

- Colab의 **런타임 → 런타임 유형 변경 → A100 GPU**를 선택하세요. A100이 배정되지 않으면 코드가 중단합니다.
- seed 42·43·44 각각에서 날씨 포함/제거 모델을 새로 학습합니다. 기존 Windows·A100 모델이나 no-weather seed 42 결과를 가져오지 않습니다.
- 56일 입력, 7일 출력, batch 128, BF16, 최대 20 epoch와 기존 검증 loss 기준 선택 정책을 유지합니다.
- 날씨 입력은 과거 관측과 persistence입니다. 실제 발급 예보 실험이나 날씨의 인과 효과 검증이 아닙니다.
- 코드 ZIP은 이 저장소의 코드를 포함한 전달본입니다. GitHub 로그인·API 토큰은 필요 없습니다. Drive 마운트만 본인 계정으로 승인하세요.
- 셀을 위에서부터 순서대로 실행하세요. 체크포인트와 로그는 Drive의 **새 실험 폴더**에 저장합니다. 입력은 /content로 복사합니다.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile

def sha256(path):
    result = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            result.update(chunk)
    return result.hexdigest()

def run(command, *, cwd=None, capture=False):
    return subprocess.run(
        [str(value) for value in command], cwd=cwd, check=True,
        text=True, encoding="utf-8", capture_output=capture,
    )

def gpu_identity():
    result = run([
        "nvidia-smi", "--query-gpu=name,uuid,driver_version",
        "--format=csv,noheader,nounits",
    ], capture=True)
    rows = [line.strip().split(", ") for line in result.stdout.splitlines() if line.strip()]
    if len(rows) != 1 or "A100" not in rows[0][0]:
        raise RuntimeError("하나의 A100 GPU가 필요합니다. 런타임 유형을 확인하세요.")
    return dict(zip(("name", "uuid", "driver_version"), rows[0], strict=True))

GPU = gpu_identity()
driver = tuple(int(part) for part in GPU["driver_version"].split("."))
# CUDA 13.x requires R580+. CUDA 12.x minor compatibility starts at
# Linux 525.60.13; the actual kernel probe below must also pass.
if driver[0] >= 580:
    CUDA_WHEEL = "cu130"
elif driver >= (525, 60, 13):
    CUDA_WHEEL = "cu128"
else:
    raise RuntimeError("지원 범위 밖의 NVIDIA 드라이버입니다. 새 A100 런타임이 필요합니다.")

def assert_same_gpu():
    if gpu_identity() != GPU:
        raise RuntimeError("GPU/UUID/드라이버가 바뀌었습니다. 기존 결과와 섞지 말고 새 실험을 시작하세요.")

print(json.dumps({"gpu": GPU, "selected_wheel": CUDA_WHEEL}, ensure_ascii=False, indent=2))


## 코드 ZIP 업로드

저장소 코드 ZIP **하나**를 선택하세요. 전달받은 ZIP SHA-256이 있으면 아래 EXPECTED_CODE_SHA256에 붙여 넣으세요. 빈 값이면 실제 해시를 계산하여 기록하지만 배포본과의 일치 여부까지 확인하지는 않습니다. 압축을 푼 코드는 이 실험 동안 수정하지 마세요.


In [ ]:
from google.colab import files

EXPECTED_CODE_SHA256 = ""  # 전달된 코드 ZIP의 SHA-256; 모르면 빈 문자열
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("코드 ZIP 하나만 업로드하세요.")
uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
if not uploaded_name.lower().endswith(".zip"):
    raise ValueError("코드 ZIP 파일이 필요합니다.")
CODE_ZIP = Path("/content") / Path(uploaded_name).name
CODE_ZIP.write_bytes(uploaded_bytes)
CODE_ZIP_SHA256 = sha256(CODE_ZIP)
if EXPECTED_CODE_SHA256 and CODE_ZIP_SHA256 != EXPECTED_CODE_SHA256.lower().strip():
    raise ValueError("코드 ZIP SHA-256이 전달된 값과 다릅니다.")
CODE_DIRECTORY = Path("/content/tft-paired-code")
if CODE_DIRECTORY.exists():
    raise FileExistsError("코드 폴더가 이미 있습니다. 같은 셀을 중복 실행하지 마세요.")
CODE_DIRECTORY.mkdir()
with zipfile.ZipFile(CODE_ZIP) as archive:
    destination = CODE_DIRECTORY.resolve()
    for member in archive.infolist():
        path = (destination / member.filename).resolve()
        if not path.is_relative_to(destination) or (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError("ZIP의 외부 경로나 심볼릭 링크를 허용하지 않습니다.")
    archive.extractall(destination)
candidates = list(CODE_DIRECTORY.rglob("ml/run_multiseed.py"))
if len(candidates) != 1:
    raise ValueError("ZIP에서 ml/run_multiseed.py를 하나만 찾을 수 있어야 합니다.")
REPO = candidates[0].parent.parent
required_code = [
    "ml/__init__.py", "ml/train_ablation.py", "ml/evaluate.py", "ml/requirements-inference.txt",
    "ml/notebooks/startup_thon_1.ipynb", "apps/api/app/forecasting.py",
]
for relative in required_code:
    if not (REPO / relative).is_file():
        raise FileNotFoundError(f"코드 ZIP에 필요한 파일이 없습니다: {relative}")
print(json.dumps({"code_directory": str(REPO), "code_zip_sha256": CODE_ZIP_SHA256}, indent=2))


## Drive 연결과 입력 복사

원본 폴더는 기본적으로 **내 드라이브/weather_order_tft**입니다. 계정에 실제로 보이는 경로가 다르면 DRIVE_SOURCE만 바꾸세요. 데이터 접근이 가능한 본인 Drive 계정으로 마운트합니다.

원본 3개 CSV/ZIP과 원본 실행의 설정·선정 시계열·검증/테스트 예측을 복사합니다. 기존 체크포인트는 읽거나 복사하지 않습니다. 미래 테스트 실측은 평가에만 쓰며 학습 입력으로 추가하지 않습니다.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_SOURCE = Path("/content/drive/MyDrive/weather_order_tft")
REFERENCE_RUN_ID = "walmart_20261024T020058Z"
LOCAL_INPUT = Path("/content/tft-paired-inputs")
if LOCAL_INPUT.exists():
    raise FileExistsError("입력 복사 폴더가 이미 있습니다. 원본을 조용히 덮어쓰지 않습니다.")
DATA = LOCAL_INPUT / "data"
REFERENCE = LOCAL_INPUT / "reference"
DATA.mkdir(parents=True)
REFERENCE.mkdir()
COPIED_INPUTS = {}
for name in ("train.csv", "weather.csv", "key.csv"):
    matches = [path for path in (DRIVE_SOURCE / name, DRIVE_SOURCE / (name + ".zip")) if path.is_file()]
    if len(matches) != 1:
        raise ValueError(f"{name} 또는 {name}.zip 하나가 필요합니다: {DRIVE_SOURCE}")
    source = matches[0]
    destination = DATA / source.name
    shutil.copyfile(source, destination)
    if sha256(source) != sha256(destination):
        raise ValueError(f"복사 중 내용이 달라졌습니다: {source.name}")
    COPIED_INPUTS[name] = {"filename": destination.name, "bytes": destination.stat().st_size, "sha256": sha256(destination)}
for name in ("run_config.json", "selected_series.csv", "validation_predictions.csv", "test_predictions.csv"):
    source = DRIVE_SOURCE / "runs" / REFERENCE_RUN_ID / name
    destination = REFERENCE / name
    shutil.copyfile(source, destination)
    if sha256(source) != sha256(destination):
        raise ValueError(f"원본 실행 파일 복사 검증 실패: {name}")
    COPIED_INPUTS["reference/" + name] = {"filename": name, "bytes": destination.stat().st_size, "sha256": sha256(destination)}

EXPERIMENT_NAME = "paired-tft-a100-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EXPERIMENT = Path("/content/drive/MyDrive/tft-paired-experiments") / EXPERIMENT_NAME
print(json.dumps({"local_inputs": str(LOCAL_INPUT), "new_experiment": str(EXPERIMENT),
                  "inputs": COPIED_INPUTS}, ensure_ascii=False, indent=2))


## 독립 Python 3.11 환경 설치

기본 Colab 커널은 Drive/UI용으로 유지합니다. 모델은 **Python 3.11.14**, uv 0.13.1로 만든 별도 환경에서만 실행합니다. 기존 추론 요구사항의 버전 고정을 사용하되 CPU PyTorch wheel/index를 선택한 CUDA wheel로 바꿉니다. NVIDIA 런타임 등 실제 설치된 전이 의존성은 전체 freeze로 저장합니다.

cu130은 R580 이상, cu128은 CUDA 12.x 드라이버 호환 범위에서 후보로 선택합니다. 드라이버의 버전 숫자만으로 성공을 보장하지 않으며 다음 셀의 CUDA/BF16 검사가 필수입니다. **계획 생성 후 wheel·패키지·Python을 바꾸지 마세요.** 설치가 실패하면 아직 학습은 시작되지 않습니다.


In [ ]:
assert_same_gpu()
PYTHON_VERSION = "3.11.14"
UV_VERSION = "0.13.1"
ENVIRONMENT = Path("/content/tft-paired-env")
if ENVIRONMENT.exists():
    raise FileExistsError("환경 폴더가 이미 있습니다. 실패한 환경을 자동 덮어쓰지 않습니다.")
run([sys.executable, "-m", "pip", "install", "uv==" + UV_VERSION])
UV = shutil.which("uv")
if not UV:
    raise RuntimeError("uv 실행파일을 찾을 수 없습니다.")
run([UV, "python", "install", PYTHON_VERSION])
run([UV, "venv", "--python", PYTHON_VERSION, ENVIRONMENT])
PYTHON = ENVIRONMENT / "bin/python"

requirements = (REPO / "ml/requirements-inference.txt").read_text(encoding="utf-8")
if "torch==2.11.0+cpu" not in requirements:
    raise ValueError("검토된 PyTorch 2.11.0 CPU 요구사항과 다릅니다.")
requirements = requirements.replace(
    "--extra-index-url https://download.pytorch.org/whl/cpu",
    "--extra-index-url https://download.pytorch.org/whl/" + CUDA_WHEEL,
).replace("torch==2.11.0+cpu", "torch==2.11.0+" + CUDA_WHEEL)
GPU_REQUIREMENTS = Path("/content/requirements-paired-a100.txt")
GPU_REQUIREMENTS.write_text("# Colab candidate; verified only after the runtime probe passes.\n" + requirements, encoding="utf-8")
run([UV, "pip", "install", "--python", PYTHON, "--index-strategy", "unsafe-best-match", "-r", GPU_REQUIREMENTS])
run([UV, "pip", "check", "--python", PYTHON])
FREEZE = run([UV, "pip", "freeze", "--python", PYTHON], capture=True).stdout
print(run([PYTHON, "--version"], capture=True).stdout.strip())


In [ ]:
probe = r'''
import json
import platform
from importlib.metadata import version
import torch
import pytorch_forecasting
import lightning.pytorch
import sklearn

assert torch.cuda.is_available(), "CUDA unavailable"
assert torch.cuda.device_count() == 1, "Expected one GPU"
assert "A100" in torch.cuda.get_device_name(0), "Expected an A100"
assert torch.cuda.is_bf16_supported(), "BF16 unavailable"
torch.manual_seed(42)
model = torch.nn.Sequential(torch.nn.Linear(32, 64), torch.nn.ReLU(), torch.nn.Linear(64, 1)).cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    prediction = model(torch.randn(128, 32, device="cuda"))
    loss = prediction.square().mean()
assert prediction.dtype == torch.bfloat16
loss.backward()
assert torch.isfinite(loss)
assert all(torch.isfinite(p.grad).all() for p in model.parameters())
optimizer.step()
torch.cuda.synchronize()
packages = ["torch", "pytorch-forecasting", "lightning", "pytorch-lightning",
            "torchmetrics", "pandas", "numpy", "scikit-learn"]
print(json.dumps({"python": platform.python_version(), "cuda_runtime": torch.version.cuda,
                  "device": torch.cuda.get_device_name(0), "precision": "bf16-mixed",
                  "bf16_forward_backward_probe": "passed", "packages": {name: version(name) for name in packages}}))
'''
RUNTIME = json.loads(run([PYTHON, "-c", probe], capture=True).stdout)
if RUNTIME["python"] != PYTHON_VERSION or RUNTIME["packages"]["torch"] != "2.11.0+" + CUDA_WHEEL:
    raise RuntimeError("실제 설치된 Python/PyTorch가 선택한 환경과 다릅니다.")
assert_same_gpu()
RUNTIME_EVIDENCE = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "gpu": GPU, "cuda_wheel": CUDA_WHEEL, "uv_version": UV_VERSION,
    "code_zip_sha256": CODE_ZIP_SHA256, "runtime": RUNTIME,
    "inputs_copied_to_content": COPIED_INPUTS,
    "fresh_model_fits": 6, "reused_model_fits": 0,
}
print(json.dumps(RUNTIME_EVIDENCE, ensure_ascii=False, indent=2))


## 1. 새 실험 계획 생성

기존 실험 폴더와 계획은 사용하지 않습니다. 아래 명령은 코드·원본 노트북을 고정하고 입력 해시와 환경을 기록합니다. **재사용 옵션을 전달하지 않습니다.** 모델·분할·batch·정밀도 정책은 기존 실행 코드가 검증합니다.


In [ ]:
assert_same_gpu()
if EXPERIMENT.exists():
    raise FileExistsError("새 실험 폴더가 필요합니다. 기존 계획/결과를 덮어쓰지 않습니다.")
run([
    PYTHON, "-u", "-m", "ml.run_multiseed", "plan",
    "--data-dir", DATA, "--reference-run", REFERENCE,
    "--output-dir", EXPERIMENT, "--python", PYTHON,
], cwd=REPO)
PLAN = EXPERIMENT / "experiment-plan.json"
plan = json.loads(PLAN.read_text(encoding="utf-8"))
assert len(plan["runs"]) == 6 and all(not item["reuse"] for item in plan["runs"])
assert plan["seeds"] == [42, 43, 44] and plan["precision"] == "bf16-mixed"
assert plan["training_policy"]["BATCH_SIZE"] == 128
assert "A100" in plan["device"]
(EXPERIMENT / "colab-runtime.json").write_text(
    json.dumps(RUNTIME_EVIDENCE, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
(EXPERIMENT / "requirements-installed.txt").write_text(FREEZE, encoding="utf-8")
shutil.copyfile(GPU_REQUIREMENTS, EXPERIMENT / GPU_REQUIREMENTS.name)
(EXPERIMENT / "source-code-zip-sha256.txt").write_text(CODE_ZIP_SHA256 + "\n", encoding="utf-8")
print(json.dumps({"plan": str(PLAN), "plan_sha256": sha256(PLAN), "fresh_runs": 6}, indent=2))


## 2. 여섯 모델 순차 학습

한 번에 GPU 학습 하나만 실행합니다. 실행기는 모델별 로그와 status.json을 Drive에 남깁니다. CPU 재현 검증이나 웹/API 서버를 실행하지 않습니다.

Colab 자원 제한·연결 해제로 중단될 수 있습니다. **중단된 모델 출력 폴더는 자동 재개/덮어쓰기가 불가능합니다.** 환경이나 GPU UUID가 바뀌면 새 실험 폴더에서 여섯 번을 다시 시작하고 이전 결과와 혼합하지 마세요. 같은 세션에서 완료 모델만 있는 경우 실행기는 완료 증거를 검증한 뒤 건너뛸 수 있지만, 이번 최초 실행은 여섯 개 모두 새 학습입니다.


In [ ]:
assert_same_gpu()
if not PLAN.is_file():
    raise FileNotFoundError("먼저 새 실험 계획을 생성하세요.")
run([PYTHON, "-u", "-m", "ml.run_multiseed", "run", "--plan", PLAN], cwd=REPO)
state = json.loads((EXPERIMENT / "status.json").read_text(encoding="utf-8"))
if state.get("status") != "all_six_completed":
    raise RuntimeError("여섯 모델이 모두 완료되지 않았습니다. 결과를 집계하지 않습니다.")
print(json.dumps({"status": state["status"], "completed_runs": len(state["completed"])}, indent=2))


## 3. 완료된 여섯 모델만 집계

전체 7,973행과 실제 56일 이력이 연속으로 존재하는 1,701행을 각각 같은 행끼리 비교합니다. API 요일 기준선은 실제 함수와 84일 관측 이력으로 계산합니다. 세 seed의 평균·표본 표준편차(ddof=1)를 기록하며, 과거 발표의 단일 실행 표를 덮어쓰지 않습니다.


In [ ]:
assert_same_gpu()
state = json.loads((EXPERIMENT / "status.json").read_text(encoding="utf-8"))
if state.get("status") != "all_six_completed" or len(state.get("completed", [])) != 6:
    raise RuntimeError("완료된 여섯 실행이 필요합니다.")
SUMMARY = EXPERIMENT / "summary"
if SUMMARY.exists():
    raise FileExistsError("요약 폴더가 이미 있습니다. 기존 평가표를 덮어쓰지 않습니다.")
run([PYTHON, "-u", "-m", "ml.run_multiseed", "summarize",
     "--plan", PLAN, "--output-dir", SUMMARY], cwd=REPO)
for name in ("metrics.csv", "api-baseline.csv", "paired-differences.csv", "summary.csv"):
    path = SUMMARY / name
    print("\n" + name + "  SHA256=" + sha256(path))
    print(path.read_text(encoding="utf-8"))
print("전체 결과/로그/체크포인트:", EXPERIMENT)


## 결과를 전달할 때

Drive의 experiment-plan.json, colab-runtime.json, requirements-installed.txt와 summary/를 함께 보관하세요. 6개 실행의 원본 예측·protocol·completion·체크포인트도 같은 실험 폴더에 남아 있습니다. 큰 원본/체크포인트를 저장소에 자동 업로드하지 않습니다.

세 seed 결과는 새 A100 환경의 비교입니다. Windows 결과 또는 과거 단일 A100 수치와 한 표의 seed로 섞지 마세요. 이번 실험의 실제 새 실행 여부는 **experiment-plan.json의 6개 reuse:false**와 **colab-runtime.json의 reused_model_fits:0**로 확인합니다.

참고: [PyTorch 공식 이전 버전 설치](https://pytorch.org/get-started/previous-versions/), [CUDA 드라이버 호환 범위](https://docs.nvidia.com/deploy/cuda-compatibility/minor-version-compatibility.html). CUDA 12 minor compatibility는 모든 PTX/기능의 실행을 보장하지 않으므로 실제 사전 검사와 학습 로그를 확인해야 합니다.
